In [9]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from pybaseball import playerid_reverse_lookup
from pybaseball import statcast
from pybaseball import playerid_lookup
import openpyxl
import pickle
from pathlib import Path
import os
import re
import time
from datetime import datetime, timedelta
from pybaseball import statcast_single_game, schedule_and_record, pitching_stats_range, batting_stats_range, statcast_pitcher

globalYear = 2010
globalMonth = 4


In [10]:
batting_columns=['team','league','ab','runs','hits','doub','trip','hr','rbi','bb','avg','obp','slg',
                'est_ba_sa','est_woba_sa','sum_woba']
reliever_columns=['team_relief','league_relief','innings','hits','bb',
                      'k','at_bats','doub','trip','hr','era','ba',
                     'slg','obp','est_ba_sa','est_woba_sa','sum_woba']
def team_abreviator(team,league):
    if team=='Atlanta':
        return 'ATL'
    elif team=='Arizona':
        return 'ARI'
    elif team=='Baltimore':
        return 'BAL'
    elif team=='Boston':
        return 'BOS'
    elif team=='Chicago':
        if league=='MLB-AL':
            return 'CWS'
        else:
            return 'CHC'
    elif team=='Cincinnati':
        return 'CIN'
    elif team=='Cleveland':
        return 'CLE'
    elif team=='Colorado':
        return 'COL'
    elif team=='Detroit':
        return 'DET'
    elif team=='Houston':
        return 'HOU'
    elif team=='Kansas City':
        return 'KC'
    elif team=='Los Angeles':
        if league=='MLB-AL':
            return 'LAA'
        else:
            return 'LAD'
    elif team=='Minnesota':
        return 'MIN'
    elif team=='Milwaukee':
        return 'MIL'
    elif team=='Miami':
        return 'MIA'
    elif team=='New York':
        if league=='MLB-AL':
            return 'NYY'
        else:
            return 'NYM'
    elif team=='Oakland':
        return 'OAK'
    elif team=='Pittsburgh':
        return 'PIT'
    elif team=='Philadelphia':
        return 'PHI'
    elif team=='San Diego':
        return 'SD'
    elif team=='San Francisco':
        return 'SF'
    elif team=='Seattle':
        return 'SEA'
    elif team=='St. Louis':
        return 'STL'
    elif team=='Texas':
        return 'TEX'
    elif team=='Tampa Bay':
        return 'TB'
    elif team=='Toronto':
        return 'TOR'
    elif team=='Washington':
        return 'WSH'
    else:
        print('Team not available in every pitch data')
        return 0

#####
def recent_team_batting(data,every_pitch,start,end,team,league):
    data=data[data.iloc[:,4]==team]
    data=data[data.iloc[:,3]==league]
    ab=data.AB.sum()
    hits=data.H.sum()
    bb=data.BB.sum()+data.IBB.sum()+data.HBP.sum()
    doub=data['2B'].sum()
    trip=data['3B'].sum()
    hr=data.HR.sum()
    rbi=data.RBI.sum()
    avg=round(hits/ab,3)
    obp=round((hits+bb)/ab,3)
    slg=round((hits+doub+(trip*2)+(hr*3))/ab,3)
    runs=data.R.sum()
    data2=every_pitch[every_pitch.game_date<=end]
    data2=data2[data2.game_date>=start]
    team_abrev=team_abreviator(team,league)
    data3=data2[data2.home_team==team_abrev]
    data3=data3[data3.inning_topbot=='Bot']
    data4=data2[data2.away_team==team_abrev]
    data4=data4[data4.inning_topbot=='Top']
    data5=pd.concat([data3,data4])
    data5=data5.dropna(subset=['launch_angle', 'launch_speed', 'estimated_ba_using_speedangle'])
    est_ba_sa=data5.estimated_ba_using_speedangle.mean()
    est_woba_sa=data5.estimated_woba_using_speedangle.mean()
    sum_woba=data5.woba_value.sum()
    df=pd.DataFrame([[team,league,ab,runs,hits,doub,trip,hr,rbi,bb,avg,obp,slg,
                     est_ba_sa,est_woba_sa,sum_woba]],columns=batting_columns)
    return df
#####
def recent_bullpen(data,every_pitch,team,league,lookback_start,lookback_end):
    pen=data[data.Tm==team]
    pen=pen[pen.Lev==league]
    pen.IP=((pen.IP-round(pen.IP,0))*(10/3))+(round(pen.IP,0))
    innings=pen.IP.sum()
    hits=pen.H.sum()*9/innings
    bb=(pen.BB.sum()+pen.HBP.sum()+pen.IBB.sum())*9/innings
    k=pen.SO.sum()*9/innings
    at_bats=pen.AB.sum()*9/innings
    doub=pen['2B'].sum()*9/innings
    trip=pen['3B'].sum()*9/innings
    hr=pen.HR.sum()*9/innings
    era=pen.ER.sum()/innings*9
    ba=hits/at_bats
    slg=round((hits+doub+(trip*2)+(hr*3))/at_bats,3)
    obp=round((hits+bb)/(at_bats+bb),3)
    data2=every_pitch[every_pitch.game_date<=lookback_end]
    data2=data2[data2.game_date>=lookback_start]
    team_abrev=team_abreviator(team,league)
    data3=data2[data2.home_team==team_abrev]
    data3=data3[data3.inning_topbot=='Bot']
    data4=data2[data2.away_team==team_abrev]
    data4=data4[data4.inning_topbot=='Top']
    data5=pd.concat([data3,data4])
    data5=data5.dropna(subset=['launch_angle', 'launch_speed', 'estimated_ba_using_speedangle'])
    est_ba_sa=data5.estimated_ba_using_speedangle.mean()
    est_woba_sa=data5.estimated_woba_using_speedangle.mean()
    sum_woba=data5.woba_value.sum()/innings
    df=pd.DataFrame([[team,league,innings,hits,bb,
                      k,at_bats,doub,trip,hr,era,ba,
                     slg,obp,est_ba_sa,est_woba_sa,sum_woba]],columns=reliever_columns)
    return df
#####

In [11]:
def get_game_data_range_local():
    listOfEveryPitchFilenames = []

    # List of directories to search
    directories = [
        r"D:\BaseballBetsData1",
        r"D:\BaseballBetsData2",
        r"D:\BaseballBetsData3",
        r"D:\BaseballBetsData4",
        r"D:\BaseballBetsData5",
        r"D:\BaseballBetsData6",
        r"D:\BaseballBetsData7"
    ]

    month_str = f"{globalMonth:02d}"  # Pads single digits with a leading zero
    pattern = rf"{globalYear}_every_pitch_{month_str}_\d{{2}}\.pkl$"

    # Iterate through each specified directory
    for directory in directories:
        if os.path.exists(directory):
            for filename in os.listdir(directory):
                if re.search(pattern, filename):
                    listOfEveryPitchFilenames.append(os.path.join(directory, filename))

    print(f"Found {len(listOfEveryPitchFilenames)} files.")
    return listOfEveryPitchFilenames

In [12]:
def get_all_starters(data,temp_every_game):
    all_starters={}

    '''
    # Extracting dates from filenames
    splits = [filename.split('_') for filename in temp_every_game]

    # Concatenating the parts of the date and formatting it
    formatSplit = ['{}-{}-{}'.format(date[0], date[3], date[4][0:2]) for date in splits]

    dates = pd.to_datetime(formatSplit)
    '''

    # Extracting dates from filenames
    splits = [filename.split('_') for filename in temp_every_game]

    # Concatenating the parts of the date and formatting it
    formatSplit = ['{}-{}-{}'.format(date[0][-4:], date[3], date[4][0:2]) for date in splits]

    # Convert to datetime
    dates = pd.to_datetime(formatSplit)


    for day in dates:
        print("get_all_starters day", day)
        day_starters=[]
        day_data=data[data.game_date==day]
        today_games=day_data.game_pk.unique()
        for game in today_games:
            print("get_all_starters game", game)
            game_stats=day_data[day_data.game_pk==game]
            home_counter=0
            away_counter=0
            l=game_stats.pitcher.value_counts().keys()
            for pitcher in l:
                m=game_stats[game_stats.pitcher==pitcher]
                m.reset_index(drop=True, inplace=True)
                if home_counter==0 and m.inning_topbot[0]=='Bot':
                    home_starter_id=pitcher
                    home_counter+=1
                elif away_counter==0 and m.inning_topbot[0]=='Top':
                    away_starter_id=pitcher
                    away_counter=+1
                else:
                    None
            home_holder=all_players[all_players.key_mlbam==home_starter_id]
            home_holder.reset_index(drop=True,inplace=True)
            home_starter_name=str(home_holder.name_first[0])+' '+str(home_holder.name_last[0])
            away_holder=all_players[all_players.key_mlbam==away_starter_id]
            away_holder.reset_index(drop=True,inplace=True)
            away_starter_name=str(away_holder.name_first[0])+' '+str(away_holder.name_last[0])
            day_starters.append(home_starter_name)
            day_starters.append(away_starter_name)
        exit_date=day.strftime('%Y-%m-%d')
        all_starters.update({exit_date:day_starters})
    return all_starters

def get_all_relievers(starters_on_day,data,temp_every_game):
    all_starters=[]
    # Extracting dates from filenames
    splits = [filename.split('_') for filename in temp_every_game]

    # Concatenating the parts of the date and formatting it
    formatSplit = ['{}-{}-{}'.format(date[0][-4:], date[3], date[4][0:2]) for date in splits]

    dates = pd.to_datetime(formatSplit)

    for day in formatSplit:
        print("get_all_relievers day", day)
        for player in starters_on_day[day]:
            print("get_all_relievers player", player)
            all_starters.append(player)
    f=pd.DataFrame(all_starters,columns=['starters'])
    f=f.starters.unique()
    g=pd.DataFrame(f,columns=['starters'])
    h=pd.merge(data,g,how='outer',left_on='Name',right_on='starters',indicator=True)
    relievers=h[h._merge!='both']
    return relievers

def fetch_data_every_game(game):
    game_ids=[]
    all_pitching_stats=[]
    all_batting_stats=[]
    data=pd.DataFrame([])
    fails=pd.DataFrame([])
    data2=pd.DataFrame([])
    home_starter_stats=pd.DataFrame([])
    away_starter_stats=pd.DataFrame([])
    
    home_batting_stats=pd.DataFrame([],columns=batting_columns)
    away_batting_stats=pd.DataFrame([],columns=batting_columns)
    home_reliever_stats=pd.DataFrame([],columns=reliever_columns)
    away_reliever_stats=pd.DataFrame([],columns=reliever_columns)
    
    all_starting_pitchers=[]
    # harrison check
    '''
    today_games_pickle_in=open(game,"rb")
    today_games=pickle.load(today_games_pickle_in)
    '''

    directories = [
        r"D:\BaseballBetsData1",
        r"D:\BaseballBetsData2",
        r"D:\BaseballBetsData3",
        r"D:\BaseballBetsData4",
        r"D:\BaseballBetsData5",
        r"D:\BaseballBetsData6",
        r"D:\BaseballBetsData7"
    ]

    # Iterate through each specified directory
    for directory in directories:
        full_path = os.path.join(directory, game)
        if os.path.exists(full_path):
            with open(full_path, "rb") as today_games_pickle_in:
                 today_games=pickle.load(today_games_pickle_in)
    
    print("game", game)

    '''
    temp_all_pitching_stats_fn = game[:4] + "_all_pitching_stats_" + game[17:19] + "_" + game[20:22] + ".pkl"
    all_pitching_stats=open(temp_all_pitching_stats_fn,"rb")
    all_pitching_stats=pickle.load(all_pitching_stats)
    '''
    
    # Extracting year, month, and day
    parts = game.split('_')
    year = parts[0].split('\\')[-1]  # Get '2019'
    month = parts[3]  # Get '04'
    day = parts[4][:2]  # Get '01'

    # Creating the new string
    temp_all_pitching_stats_fn = f"{year}_all_pitching_stats_{month}_{day}.pkl"


    #temp_all_pitching_stats_fn = game[0][-4:] + "_all_pitching_stats_" + game[17:19] + "_" + game[20:22] + ".pkl"
    print("hi harry", temp_all_pitching_stats_fn)

    # List of directories to search
    directories = [
        r"D:\BaseballBetsData1",
        r"D:\BaseballBetsData2",
        r"D:\BaseballBetsData3",
        r"D:\BaseballBetsData4",
        r"D:\BaseballBetsData5",
        r"D:\BaseballBetsData6",
        r"D:\BaseballBetsData7"
    ]

    # Iterate through each specified directory
    for directory in directories:
        full_path = os.path.join(directory, temp_all_pitching_stats_fn)
        print("full_path.......................", directory, temp_all_pitching_stats_fn)
        if os.path.exists(full_path):
            with open(full_path, "rb") as all_pitching_stats:
                print("this ran.......................", full_path)
                all_pitching_stats=pickle.load(all_pitching_stats)


    if len(all_pitching_stats) == 0:
        return data,fails
    print("all_pitching_stats2", all_pitching_stats)
    all_pitching_stats.Name=all_pitching_stats.Name.str.lower()

    all_reliever_stats=get_all_relievers(starters_on_day,all_pitching_stats, every_game)

    print("all_reliever_stats", all_reliever_stats)

    '''
    temp_all_batting_stats_fn = game[:4] + "_all_batting_stats_" + game[17:19] + "_" + game[20:22] + ".pkl"
    all_batting_stats=open(temp_all_batting_stats_fn,"rb")
    all_batting_stats=pickle.load(all_batting_stats)
    '''

    #temp_all_batting_stats_fn = game[:4] + "_all_batting_stats_" + game[17:19] + "_" + game[20:22] + ".pkl"

    # Extracting year, month, and day
    parts = game.split('_')
    year = parts[0].split('\\')[-1]  # Get '2019'
    month = parts[3]  # Get '04'
    day = parts[4][:2]  # Get '01'

    # Creating the new string
    temp_all_batting_stats_fn = year + "_all_batting_stats_" + month + "_" + day + ".pkl"

    print("temp_all_batting_stats_fn", temp_all_batting_stats_fn)

    # List of directories to search
    directories = [
        r"D:\BaseballBetsData1",
        r"D:\BaseballBetsData2",
        r"D:\BaseballBetsData3",
        r"D:\BaseballBetsData4",
        r"D:\BaseballBetsData5",
        r"D:\BaseballBetsData6",
        r"D:\BaseballBetsData7"
    ]

    # Iterate through each specified directory
    for directory in directories:
        full_path = os.path.join(directory, temp_all_batting_stats_fn)
        if os.path.exists(full_path):
            with open(full_path, "rb") as all_batting_stats:
                all_batting_stats=pickle.load(all_batting_stats)

    game_ids=today_games.game_pk.unique()
    game_ids=game_ids.astype(int)
    # This deals with the occasional occurance of double headers
    double_header_count={'BOS':0,'MIL':0,'PIT':0,'MIA':0,'ATL':0,'PHI':0,
                      'CIN':0,'TOR':0,'ARI':0,'TEX':0,'OAK':0,'SF':0,
                      'LAD':0,'SD':0,'WSN':0,'NYM':0,'COL':0,'KC':0,
                      'CHW':0,'HOU':0,'BAL':0,'DET':0,'MIN':0,'CLE':0,
                      'NYY':0,'CHC':0,'STL':0,'BOS':0,'TB':0,'TB':0,
                      'LAA':0,'SEA':0}

    '''
    listOfEveryGameFilenames = []
    path = "./"
    gameStatsPattern = rf"{game[:4]}_game_stats_{game[17:19]}_{game[20:22]}_\d+\.pkl$"

    for filename in os.listdir(path):
        if re.search(gameStatsPattern, filename):
            listOfEveryGameFilenames.append(filename)
    print(len(listOfEveryGameFilenames))

    '''

    listOfEveryGameFilenames = []

    # List of directories to search
    directories = [
        r"D:\BaseballBetsData1",
        r"D:\BaseballBetsData2",
        r"D:\BaseballBetsData3",
        r"D:\BaseballBetsData4",
        r"D:\BaseballBetsData5",
        r"D:\BaseballBetsData6",
        r"D:\BaseballBetsData7"
    ]

    # Construct the regex pattern based on the game string
    #gameStatsPattern = rf"{game[:4]}_game_stats_{game[17:19]}_{game[20:22]}_\d+\.pkl$"

        # Extracting year, month, and day
    parts = game.split('_')
    year = parts[0].split('\\')[-1]  # Get '2019'
    month = parts[3]  # Get '04'
    day = parts[4][:2]  # Get '01'

    # Creating the new string
    gameStatsPattern = rf"{year}_game_stats_{month}_{day}_\d+\.pkl$"


    # Iterate through each specified directory
    for directory in directories:
        if os.path.exists(directory):
            for filename in os.listdir(directory):
                #invalid group
                if re.search(gameStatsPattern, filename):
                    listOfEveryGameFilenames.append(os.path.join(directory, filename))

    print(f"Found {len(listOfEveryGameFilenames)} game files.")



    for gameStatFn in listOfEveryGameFilenames:
        '''
        game_stats=open(gameStatFn,"rb")
        game_stats=pickle.load(game_stats)
        '''

        directories = [
            r"D:\BaseballBetsData1",
            r"D:\BaseballBetsData2",
            r"D:\BaseballBetsData3",
            r"D:\BaseballBetsData4",
            r"D:\BaseballBetsData5",
            r"D:\BaseballBetsData6",
            r"D:\BaseballBetsData7"
        ]

        # Iterate through each specified directory
        for directory in directories:
            full_path = os.path.join(directory, gameStatFn)
            if os.path.exists(full_path):
                with open(full_path, "rb") as game_stats:
                    game_stats=pickle.load(game_stats)

        home,away=game_stats.home_team[0],game_stats.away_team[0]
        home_counter=0
        away_counter=0
        l=game_stats.pitcher.value_counts().keys()
        #determine 'starter' by who threw the most pitches. Normally this would simply be the
        #pitchers who were in the first inning but with the rise of 'bullpenning' this is a work-around
        for pitcher in l:
            m=game_stats[game_stats.pitcher==pitcher]
            m.reset_index(drop=True, inplace=True)
            if home_counter==0 and m.inning_topbot[0]=='Top':
                home_starter_id=pitcher
                home_counter+=1
            elif away_counter==0 and m.inning_topbot[0]=='Bot':
                away_starter_id=pitcher
                away_counter=+1
            else:
                None
        home_holder=all_players[all_players.key_mlbam==home_starter_id]
        home_holder.reset_index(drop=True,inplace=True)
        home_starter_name=str(home_holder.name_first[0])+' '+str(home_holder.name_last[0])
        away_holder=all_players[all_players.key_mlbam==away_starter_id]
        away_holder.reset_index(drop=True,inplace=True)
        away_starter_name=str(away_holder.name_first[0])+' '+str(away_holder.name_last[0])
        # This set of if statements handles cases where a starting pitcher does not have sufficient recent
        # data to be useful in the model. Such as not having pitched in a while or the rare case where there
        # are two pitchers with the same name.
        if len(all_pitching_stats[all_pitching_stats.Name.str.contains(home_holder.name_last[0],regex=False)])==0:
            print('game#: ',game, 'Home: ',home,' Away: ',away,'home pitcher with insuffucicient history')
            fails = pd.concat([fails, pd.DataFrame({'game#':game, 'Home':home,'Away':away,
                                             'Reason':'home pitcher with insuffucicient history'},
                                            index=[0])], ignore_index=True)
            '''
                        fails=fails.append(pd.DataFrame({'game#':game, 'Home':home,'Away':away,
                                             'Reason':'home pitcher with insuffucicient history'},
                                            index=[0]),ignore_index=True)
            '''
            continue
        elif len(all_pitching_stats[all_pitching_stats.Name.str.contains(home_holder.name_last[0],regex=False)])==1:
            home_starter_stats=pd.concat([home_starter_stats, pd.DataFrame(all_pitching_stats[all_pitching_stats.Name.str.contains(home_holder.name_last[0],regex=False)])])
            #home_starter_stats=home_starter_stats.append(pd.DataFrame(all_pitching_stats[all_pitching_stats.Name.str.contains(home_holder.name_last[0],regex=False)]))
        elif len(all_pitching_stats[all_pitching_stats.Name==home_starter_name])==1:
            #home_starter_stats=home_starter_stats.append(pd.DataFrame(all_pitching_stats[all_pitching_stats.Name==home_starter_name]))
            home_starter_stats = pd.concat([home_starter_stats, pd.DataFrame(all_pitching_stats[all_pitching_stats.Name==home_starter_name])])
        else:
            print('game#: ',game,' Home: ',home,' Away: ',away,'home pitcher with duplicate name?')
            fails = pd.concat([fails, pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'home pitcher with duplicate name?'},
                                            index=[0])], ignore_index=True)
            '''
                        fails=fails.append(pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'home pitcher with duplicate name?'},
                                            index=[0]),ignore_index=True)
            '''
            None
        if len(all_pitching_stats[all_pitching_stats.Name.str.contains(away_holder.name_last[0],regex=False)])==0:
            print('game#: ',game,' Home: ',home,' Away: ',away,'away pitcher with insuffucicient history')
            fails = pd.concat([fails, pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'away pitcher with insuffucicient history'},
                                            index=[0])], ignore_index=True)
            '''
                        fails=fails.append(pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'away pitcher with insuffucicient history'},
                                            index=[0]),ignore_index=True)
            '''
            continue
        elif len(all_pitching_stats[all_pitching_stats.Name.str.contains(away_holder.name_last[0],regex=False)])==1:
            #away_starter_stats=away_starter_stats.append(pd.DataFrame(all_pitching_stats[all_pitching_stats.Name.str.contains(away_holder.name_last[0],regex=False)]))
            away_starter_stats = pd.concat([away_starter_stats, pd.DataFrame(all_pitching_stats[all_pitching_stats.Name.str.contains(away_holder.name_last[0],regex=False)])])
        elif len(all_pitching_stats[all_pitching_stats.Name==away_starter_name])==1:
            #away_starter_stats=away_starter_stats.append(pd.DataFrame(all_pitching_stats[all_pitching_stats.Name==away_starter_name]))
            away_starter_stats = pd.concat([away_starter_stats, pd.DataFrame(all_pitching_stats[all_pitching_stats.Name==away_starter_name])])
        else:
            print('game#: ',game,' Home: ',home,' Away: ',away,'away pitcher with duplicate name?')
            fails = pd.concat([fails, pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'away pitcher with duplicate name?'},
                                            index=[0])], ignore_index=True)
            '''
                        fails=fails.append(pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'away pitcher with duplicate name?'},
                                            index=[0]),ignore_index=True)
            '''
            None
        if away_starter_stats.empty or home_starter_stats.empty:
            continue
        # In different databases, there are a couple teams with different abreviations: this handles that.
        if home=='CWS':
            home='CHW'
        else:
            None
        if away=='CWS':
            away='CHW'
        else:
            None
        if home=='AZ':
            home='ARI'
        else:
            None
        if away=='AZ':
            away='ARI'
        else:
            None
        if home=='WSH':
            home='WSN'
        else:
            None
        if away=='WSH':
            away='WSN'
        else:
            None
        print(home,away)
        # This section is a workaround resulting because one database does not account for
        # extra inning games and leaves such games as a tie. To get around this I had to call
        # a different database and determine the winner by looking at the change in the team's
        # record after the game.
        '''
        home_schedule_record_fn = game[:4] + "_schedule_record_" + home + ".pkl"
        home_schedule_record=open(home_schedule_record_fn,"rb")
        home_schedule_record=pickle.load(home_schedule_record)

        '''

        # Extracting year, month, and day
        parts = game.split('_')
        year = parts[0].split('\\')[-1]  # Get '2019'

        home_schedule_record_fn = year + "_schedule_record_" + home + ".pkl"

        home_schedule_record = None
        away_schedule_record = None

        # List of directories to search
        directories = [
            r"D:\BaseballBetsData1",
            r"D:\BaseballBetsData2",
            r"D:\BaseballBetsData3",
            r"D:\BaseballBetsData4",
            r"D:\BaseballBetsData5",
            r"D:\BaseballBetsData6",
            r"D:\BaseballBetsData7"
        ]

        # Iterate through each specified directory
        for directory in directories:
            full_path = os.path.join(directory, home_schedule_record_fn)
            print("home_schedule_record directory", directory)
            print("home_schedule_record home_schedule_record_fn", home_schedule_record_fn)
            print("home_schedule_record full_path", full_path)
            if os.path.exists(full_path):
                with open(full_path, "rb") as home_schedule_record:
                    print("home_schedule_record found", home_schedule_record)
                    home_schedule_record=pickle.load(home_schedule_record)
                    home_schedule_record.set_index('Date',inplace=True)
                break

        '''
        away_schedule_record_fn = game[:4] + "_schedule_record_" + away + ".pkl"
        away_schedule_record=open(away_schedule_record_fn,"rb")
        away_schedule_record=pickle.load(away_schedule_record)
        '''

        parts = game.split('_')
        year = parts[0].split('\\')[-1]  # Get '2019'

        away_schedule_record_fn = year + "_schedule_record_" + away + ".pkl"

        # List of directories to search
        directories = [
            r"D:\BaseballBetsData1",
            r"D:\BaseballBetsData2",
            r"D:\BaseballBetsData3",
            r"D:\BaseballBetsData4",
            r"D:\BaseballBetsData5",
            r"D:\BaseballBetsData6",
            r"D:\BaseballBetsData7"
        ]

        # Iterate through each specified directory
        for directory in directories:
            full_path = os.path.join(directory, away_schedule_record_fn)
            if os.path.exists(full_path):
                with open(full_path, "rb") as away_schedule_record:
                    away_schedule_record=pickle.load(away_schedule_record)
                    away_schedule_record.set_index('Date',inplace=True)
                break

        if away_schedule_record is None or home_schedule_record is None:
            continue

        # Extract year, month, and day from the filename using regular expressions
        match = re.search(r'(\d{4})_every_pitch_(\d{2})_(\d{2})\.pkl', game)
        year = match.group(1)
        month = match.group(2)
        day = match.group(3)

        # Create a datetime object
        date_obj = datetime(int(year), int(month), int(day))

        # Format the date as "YYYY-MM-DD"
        date_string = date_obj.strftime("%Y-%m-%d")

        # Convert the date string to a datetime object
        date = datetime.strptime(date_string, "%Y-%m-%d")
        # Calculate the date 30 days prior
        prior_date = date - timedelta(days=30)
        # Format the prior_date as YYYY-MM-DD
        prior_date_formatted = prior_date.strftime("%Y-%m-%d")
        lookback_end_date = prior_date - timedelta(days=1)
        lookback_end_date_formatted = lookback_end_date.strftime("%Y-%m-%d")
        print(date_string)
        print(prior_date_formatted)
        print("lookback_end_date_formatted", lookback_end_date_formatted)


        y,y2,y3=pd.to_datetime(date),pd.to_datetime(lookback_end_date_formatted),pd.to_datetime(prior_date_formatted)
        z,z2,z3=y.day,y2.day,y3.day
        date_string,date_string2,date_string3=str(y.strftime('%A, %b '))+str(z),str(y2.strftime('%A, %b '))+str(z2),str(y3.strftime('%A, %b '))+str(z3)
        date_string4=str(date_string)+str(' (1)')
        date_string5=str(date_string)+str(' (2)')

        if any(item==date_string for item in home_schedule_record.index):
            home_score=home_schedule_record.loc[date_string].R
            away_score=home_schedule_record.loc[date_string].RA
        else:
            if sum([item==date_string for item in home_schedule_record.index])==0 and double_header_count[home]==0:
                home_score=home_schedule_record.loc[date_string4].R
                away_score=home_schedule_record.loc[date_string4].RA
                double_header_count[home]=1
            else:
                home_score=home_schedule_record.loc[date_string5].R
                away_score=home_schedule_record.loc[date_string5].RA
        print("data in loop 1", data)
        # determine winner
        if home_score>away_score:
            home_win=1
        elif home_score<away_score:
            home_win=0
        else:
            home_win=-99
        print("date_string2", date_string2)
        print("home_schedule_record.index", home_schedule_record.index)
        if any(item==date_string2 for item in home_schedule_record.index):
            print("home record 4 ran")
            home_record=home_schedule_record.loc[date_string2]['W-L']
        else:
            for i in range(1,10):
                y4=y2-pd.to_timedelta(i,unit='D')
                y4z=y4.day
                y4zdate=str(y4.strftime('%A, %b '))+str(y4z)
                if any(item==y4zdate for item in away_schedule_record.index):
                    print("home record 3 ran")
                    home_record=away_schedule_record.loc[y4zdate]['W-L']
                    break
                else:
                    None
        if any(item==date_string3 for item in home_schedule_record.index):
            print("home record 2 ran")
            home_record_lookback=home_schedule_record.loc[date_string3]['W-L']
        else:
            for i in range(1,10):
                y4=y3-pd.to_timedelta(i,unit='D')
                y4z=y4.day
                y4zdate=str(y4.strftime('%A, %b '))+str(y4z)
                if any(item==y4zdate for item in away_schedule_record.index):
                    print("home record 1 ran")
                    home_record_lookback=away_schedule_record.loc[y4zdate]['W-L']
                    break
                else:
                    None
        if any(item==date_string2 for item in away_schedule_record.index):
            away_record=away_schedule_record.loc[date_string2]['W-L']
        else:
            for i in range(1,10):
                y4=y2-pd.to_timedelta(i,unit='D')
                y4z=y4.day
                y4zdate=str(y4.strftime('%A, %b '))+str(y4z)
                if any(item==y4zdate for item in away_schedule_record.index):
                    away_record=away_schedule_record.loc[y4zdate]['W-L']
                    break
                else:
                    None
        if any(item==date_string3 for item in away_schedule_record.index):
            away_record_lookback=away_schedule_record.loc[date_string3]['W-L']
        else:
            for i in range(1,10):
                y4=y3-pd.to_timedelta(i,unit='D')
                y4z=y4.day
                y4zdate=str(y4.strftime('%A, %b '))+str(y4z)
                if any(item==y4zdate for item in away_schedule_record.index):
                    away_record_lookback=away_schedule_record.loc[y4zdate]['W-L']
                    break
                else:
                    None
        # This section determines the season win percentage for each team
        try:
            home_record
        except NameError:
            home_record = "0-0"
        try:
            away_record
        except NameError:
            away_record = "0-0"
        try:
            print("data in loop 6", data)
            home_current_wins,home_current_losses=int(home_record.split('-')[0]),int(home_record.split('-')[1])
            print("data in loop 9", data)
            away_current_wins,away_current_losses=int(away_record.split('-')[0]),int(away_record.split('-')[1])
            home_pct = 0
            away_pct = 0
            try:
                home_pct=home_current_wins/(home_current_wins+home_current_losses)
            except:
                home_pct = 0
            try:
                away_pct=away_current_wins/(away_current_wins+away_current_losses)
            except:
                away_pct = 0
            print("data in loop 8", data)
            try:
                home_record_lookback
            except:
                home_record_lookback = "0-0"
            try:
                away_record_lookback
            except:
                away_record_lookback = "0-0"
            # This section determines the recent win percentage for each team
            home_lookback_wins,home_lookback_losses=int(home_record_lookback.split('-')[0]),int(home_record_lookback.split('-')[1])
            away_lookback_wins,away_lookback_losses=int(away_record_lookback.split('-')[0]),int(away_record_lookback.split('-')[1])
            home_recent_wins,home_recent_losses=home_current_wins-home_lookback_wins,home_current_losses-home_lookback_losses
            away_recent_wins,away_recent_losses=away_current_wins-away_lookback_wins,away_current_losses-away_lookback_losses
            try:
                home_streak=home_recent_wins/(home_recent_wins+home_recent_losses)
            except:
                home_streak = 0
            try:
                away_streak=away_recent_wins/(away_recent_wins+away_recent_losses)
            except:
                away_streak = 0
            # This section gathers some advanced statistics about the starting pitchers
            print("gameStatFn", gameStatFn)
            '''
            temp_home_statcast_pitcher_fn = year + "_statcast_pitcher_" + month + "_" + day + "_" + gameStatFn.split('_')[-1].split('.')[0] + "_home.pkl"
            temp_home_statcast_pitcher_pickle_in = open(temp_home_statcast_pitcher_fn,"rb")
            home_starter_adv=pickle.load(temp_home_statcast_pitcher_pickle_in)
            '''

            temp_home_statcast_pitcher_fn = f"{year}_statcast_pitcher_{month}_{day}_{gameStatFn.split('_')[-1].split('.')[0]}_home.pkl"

            # List of directories to search
            directories = [
                r"D:\BaseballBetsData1",
                r"D:\BaseballBetsData2",
                r"D:\BaseballBetsData3",
                r"D:\BaseballBetsData4",
                r"D:\BaseballBetsData5",
                r"D:\BaseballBetsData6",
                r"D:\BaseballBetsData7"
            ]

            # Iterate through each specified directory
            for directory in directories:
                full_path = os.path.join(directory, temp_home_statcast_pitcher_fn)
                if os.path.exists(full_path):
                    with open(full_path, "rb") as temp_home_statcast_pitcher_pickle_in:
                        home_starter_adv=pickle.load(temp_home_statcast_pitcher_pickle_in)

        except Exception as e:
            print("error collecting wins/losses", e)
            continue

        print("data in loop 7", data)
        if len(home_starter_adv)==0 or "error" in home_starter_adv:
            print('No home starter advanced stats')
            fails = pd.concat([fails, pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'No home starter advanced stats'},index=[0])], ignore_index=True)
            '''
                        fails=fails.append(pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'No home starter advanced stats'},index=[0]),ignore_index=True)
            '''
            continue
        else:
            None
        print("data in loop 2", data)
        print("launch_speed", "launch_speed" in home_starter_adv)
        print("home_starter_adv", home_starter_adv.columns)
        home_starter_launch=home_starter_adv.launch_speed.mean()
        home_starter_adv = home_starter_adv.dropna(subset=['launch_angle', 'launch_speed', 'estimated_ba_using_speedangle'])
        home_starter_est_ba_sa=home_starter_adv.estimated_ba_using_speedangle.mean()
        home_starter_est_woba_sa=home_starter_adv.estimated_woba_using_speedangle.mean()
        home_starter_sum_woba=home_starter_adv.woba_value.sum()

        '''
        temp_away_statcast_pitcher_fn = year + "_statcast_pitcher_" + month + "_" + day + "_" + gameStatFn.split('_')[-1].split('.')[0] + "_away.pkl"
        temp_away_statcast_pitcher_pickle_in = open(temp_away_statcast_pitcher_fn,"rb")
        away_starter_adv=pickle.load(temp_away_statcast_pitcher_pickle_in)
        '''

        temp_away_statcast_pitcher_fn = f"{year}_statcast_pitcher_{month}_{day}_{gameStatFn.split('_')[-1].split('.')[0]}_away.pkl"

        # List of directories to search
        directories = [
            r"D:\BaseballBetsData1",
            r"D:\BaseballBetsData2",
            r"D:\BaseballBetsData3",
            r"D:\BaseballBetsData4",
            r"D:\BaseballBetsData5",
            r"D:\BaseballBetsData6",
            r"D:\BaseballBetsData7"
        ]

        # Iterate through each specified directory
        for directory in directories:
            full_path = os.path.join(directory, temp_away_statcast_pitcher_fn)
            if os.path.exists(full_path):
                with open(full_path, "rb") as temp_away_statcast_pitcher_pickle_in:
                    away_starter_adv=pickle.load(temp_away_statcast_pitcher_pickle_in)


        if len(away_starter_adv)==0 or "error" in home_starter_adv:
            print('No away starter advanced stats')
            fails = pd.concat([fails, pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'No away starter advanced stats'},index=[0])], ignore_index=True)
            '''
                        fails=fails.append(pd.DataFrame({'game#':game,'Home':home,'Away':away,
                                             'Reason':'No away starter advanced stats'},index=[0]),ignore_index=True)
            '''
            continue
        else:
            None
        print("data in loop 3", data)
        away_starter_launch=away_starter_adv.launch_speed.mean()
        away_starter_adv = away_starter_adv.dropna(subset=['launch_angle', 'launch_speed', 'estimated_ba_using_speedangle'])
        away_starter_est_ba_sa=away_starter_adv.estimated_ba_using_speedangle.mean()
        away_starter_est_woba_sa=away_starter_adv.estimated_woba_using_speedangle.mean()
        away_starter_sum_woba=away_starter_adv.woba_value.sum()
        data = pd.concat([data, pd.DataFrame({'home_win':home_win,'home_score':home_score,
                                       'away_score':away_score,'date':date,'lookback_days':'30',
                                       'home_team':home,'away_team':away,'home_pct':home_pct,'away_pct':away_pct,
                                       'home_streak':home_streak,'away_streak':away_streak,
                                       'home_starter_name':home_starter_name,
                                       'away_starter_name':away_starter_name,'home_starter_id':home_starter_id,
                                       'away_starter_id':away_starter_id,'home_starter_launch':home_starter_launch,
                                       'home_starter_est_ba_sa':home_starter_est_ba_sa,
                                       'home_starter_est_woba_sa':home_starter_est_woba_sa,
                                       'home_starter_sum_woba':home_starter_sum_woba,
                                      'away_starter_launch':away_starter_launch,
                                      'away_starter_est_ba_sa':away_starter_est_ba_sa,
                                       'away_starter_est_woba_sa':away_starter_est_woba_sa,
                                       'away_starter_sum_woba':away_starter_sum_woba},index=[0])], ignore_index=True)
        '''
                data=data.append(pd.DataFrame({'home_win':home_win,'home_score':home_score,
                                       'away_score':away_score,'date':date,'lookback_days':'30',
                                       'home_team':home,'away_team':away,'home_pct':home_pct,'away_pct':away_pct,
                                       'home_streak':home_streak,'away_streak':away_streak,
                                       'home_starter_name':home_starter_name,
                                       'away_starter_name':away_starter_name,'home_starter_id':home_starter_id,
                                       'away_starter_id':away_starter_id,'home_starter_launch':home_starter_launch,
                                       'home_starter_est_ba_sa':home_starter_est_ba_sa,
                                       'home_starter_est_woba_sa':home_starter_est_woba_sa,
                                       'home_starter_sum_woba':home_starter_sum_woba,
                                      'away_starter_launch':away_starter_launch,
                                      'away_starter_est_ba_sa':away_starter_est_ba_sa,
                                       'away_starter_est_woba_sa':away_starter_est_woba_sa,
                                       'away_starter_sum_woba':away_starter_sum_woba},index=[0]),ignore_index=True)

        '''
        print("data in loop 5", data)
        print("home_starter_stats.columns", "Tm" in home_starter_stats)
        print("away_starter_stats.columns", "Tm" in away_starter_stats)
        print("home_starter_stats.columns", home_starter_stats.columns)
        print("away_starter_stats.columns", away_starter_stats.columns)
        all_starting_pitchers.append(home_starter_name)
        all_starting_pitchers.append(away_starter_name)
        home_starter_stats.reset_index(drop=True,inplace=True)
        away_starter_stats.reset_index(drop=True,inplace=True)
        # This section calls the other functions and gathers data about each team
        home_batting=recent_team_batting(all_batting_stats,every_pitch,prior_date_formatted, lookback_end_date_formatted,home_starter_stats.Tm[len(home_starter_stats)-1],home_starter_stats.Lev[len(home_starter_stats)-1])
        #home_batting_stats=home_batting_stats.append((pd.DataFrame(home_batting)))
        home_batting_stats = pd.concat([home_batting_stats, pd.DataFrame(home_batting)])
        away_batting=recent_team_batting(all_batting_stats,every_pitch,prior_date_formatted, lookback_end_date_formatted,away_starter_stats.Tm[len(away_starter_stats)-1],away_starter_stats.Lev[len(away_starter_stats)-1])
        #away_batting_stats=away_batting_stats.append((pd.DataFrame(away_batting)))
        away_batting_stats = pd.concat([away_batting_stats, pd.DataFrame(away_batting)])
        home_relief=recent_bullpen(all_reliever_stats,every_pitch,home_starter_stats.Tm[len(home_starter_stats)-1],
                                   home_starter_stats.Lev[len(home_starter_stats)-1],prior_date_formatted,lookback_end_date_formatted)
        home_reliever_stats = pd.concat([home_reliever_stats, pd.DataFrame(home_relief)])
        #home_reliever_stats=home_reliever_stats.append((pd.DataFrame(home_relief)))
        away_relief=recent_bullpen(all_reliever_stats,every_pitch,away_starter_stats.Tm[len(away_starter_stats)-1],
                                   away_starter_stats.Lev[len(away_starter_stats)-1],prior_date_formatted,lookback_end_date_formatted)
        #away_reliever_stats=away_reliever_stats.append((pd.DataFrame(away_relief)))
        away_reliever_stats = pd.concat([away_reliever_stats, pd.DataFrame(away_relief)])

    print("double test data", data)

    if data.empty:
        return data,fails
    data['home_join']=data.home_starter_name.str.replace(' ','')
    data['away_join']=data.away_starter_name.str.replace(' ','')
    home_starter_stats.columns=['hs'+str(col) for col in home_starter_stats.columns]
    away_starter_stats.columns=['as'+str(col) for col in away_starter_stats.columns]
    home_batting_stats.columns=['home_bat_'+str(col) for col in home_batting_stats.columns]
    away_batting_stats.columns=['away_bat_'+str(col) for col in away_batting_stats.columns]
    home_reliever_stats.columns=['homepen_'+str(col) for col in home_reliever_stats.columns]
    away_reliever_stats.columns=['awaypen_'+str(col) for col in away_reliever_stats.columns]
    home_starter_stats['hsjoin']=home_starter_stats.hsName.str.replace(' ','')
    away_starter_stats['asjoin']=away_starter_stats.asName.str.replace(' ','')
    data=data.merge(home_starter_stats,left_on='home_join',right_on='hsjoin')
    data=data.merge(away_starter_stats,left_on='away_join',right_on='asjoin')
    data=data.merge(home_batting_stats,how='left',left_on=['hsTm','hsLev'],right_on=['home_bat_team','home_bat_league'])
    data=data.merge(away_batting_stats,how='left',left_on=['asTm','asLev'],right_on=['away_bat_team','away_bat_league'])
    data=data.merge(home_reliever_stats,how='left',left_on=['hsTm','hsLev'],right_on=['homepen_team_relief','homepen_league_relief'])
    data=data.merge(away_reliever_stats,how='left',left_on=['asTm','asLev'],right_on=['awaypen_team_relief','awaypen_league_relief'])
    data.drop(['hsTm','asTm','away_join','home_join','asName','as#days',
                'asAge','asLev','hsName','hs#days','hsAge','hsLev','hsjoin',
                'asjoin','homepen_team_relief','homepen_league_relief',
               'awaypen_team_relief','awaypen_league_relief'],axis=1,inplace=True)
    data=data.drop_duplicates()
    return data,fails


In [13]:
def fetch_game_data_wrapper(every_game):
    data=pd.DataFrame([])
    fails=pd.DataFrame([])
    for game in every_game:
        day_data,day_fails=fetch_data_every_game(game)
        data=pd.concat([data,day_data],ignore_index=True)
        fails=pd.concat([fails,day_fails],ignore_index=True)
    print(data)
    print(fails)
    return data

In [14]:
every_game = get_game_data_range_local()

every_pitch = pd.DataFrame([])

for game in every_game:
    '''
    today_games_pickle_in=open(game,"rb")
    today_games=pickle.load(today_games_pickle_in)
    '''

    directories = [
        r"D:\BaseballBetsData1",
        r"D:\BaseballBetsData2",
        r"D:\BaseballBetsData3",
        r"D:\BaseballBetsData4",
        r"D:\BaseballBetsData5",
        r"D:\BaseballBetsData6",
        r"D:\BaseballBetsData7"
    ]

    # Iterate through each specified directory
    for directory in directories:
        full_path = os.path.join(directory, game)
        if os.path.exists(full_path):
            with open(full_path, "rb") as today_games_pickle_in:
                today_games=pickle.load(today_games_pickle_in)

    every_pitch = pd.concat([every_pitch, today_games], ignore_index=True)

all_players=open("wrangle_data_all_players.pkl", "rb")
all_players=pickle.load(all_players)

# use every pitch instead files instead of the hard coded dates
starters_on_day=get_all_starters(every_pitch,every_game)
print("starters_on_day",starters_on_day)

game_data = fetch_game_data_wrapper(every_game)
print("game_data formally wrapper",game_data)


# The section below transforms the dates into the form that matches the rest of this notebook
odds_data=pd.read_csv('mlb2019odds_june23.csv')
odds_data['month']=round(odds_data.Date/100).astype(int)
odds_data['day']=(odds_data.Date-odds_data.month*100).astype(int)
odds_data['game_day']=0
# This corrects team abreviations from the odds_data file
for i in range(len(odds_data)):
    if odds_data.Team[i]=='SFO':
        odds_data.Team[i]='SF'
    elif odds_data.Team[i]=='WAS':
        odds_data.Team[i]='WSN'
    elif odds_data.Team[i]=='TAM':
        odds_data.Team[i]='TB'
    elif odds_data.Team[i]=='CWS':
        odds_data.Team[i]='CHW'
    elif odds_data.Team[i]=='KAN':
        odds_data.Team[i]='KC'
    elif odds_data.Team[i]=='CUB':
        odds_data.Team[i]='CHC'
    elif odds_data.Team[i]=='SDG':
        odds_data.Team[i]='SD'
    else:
        None
    month=odds_data.month[i]
    day=odds_data.day[i]
    odds_data['game_day'][i]=datetime(globalYear,month,day).strftime('%Y-%m-%d')
# This initiates the necessary variables
game_data['home_money_open']=None
game_data['home_money_close']=None
game_data['home_money_change']=None
game_data['away_money_open']=None
game_data['away_money_close']=None
game_data['away_money_change']=None
game_data['home_prob_open']=None
game_data['home_prob_close']=None
game_data['home_prob_change']=None
game_data['ou_open']=None
game_data['ou_close']=None
# This function uses the money line odds to calculate the betting markets implied
# probablility of the home team winning
def home_pct_chance(home_money,away_money):
    if home_money>0:
        a1=100/(home_money+100)
    else:
        a1=-home_money/(100-home_money)
    if away_money>0:
        a2=100/(away_money+100)
    else:
        a2=-away_money/(100-away_money)
    return(a1/(a1+a2))
# This for loop fills in the relevant betting related columns into the data frame
print("this ran game_data", game_data)
for i in range(len(game_data)):
    #try:
    home=[]
    away=[]
    date=game_data.date[i].strftime('%Y-%m-%d')
    home_team=game_data.home_team[i]
    away_team=game_data.away_team[i]
    home=odds_data.loc[(odds_data['game_day'] == date) & (odds_data['Team'] == home_team)]
    away=odds_data.loc[(odds_data['game_day'] == date) & (odds_data['Team'] == away_team)]
    if odds_data.empty or home.empty or away.empty:
        break
    print("The DataFrame is empty.")
    print("odds_data 2", odds_data)
    print("odds home", home)
    print("odds away", away)
    print("date", date)
    print("odds_data['game_day']", odds_data['game_day'][0])
    print("home_team", home_team)
    print("away_team", away_team)
    home.reset_index(drop=True,inplace=True)
    away.reset_index(drop=True,inplace=True)
    print("in loop before game_data.home_money_close[i]=int(home.Close[0]) 4")
    print(home)
    game_data.home_money_close[i]=int(home.Close[0])
    print("in loop before conditions")
    if home.Open[0]=='NL':
        game_data.home_money_open[i]=game_data.home_money_close[i]
        game_data.home_money_change[i]=0
    else:
        game_data.home_money_open[i]=int(home.Open[0])
        game_data.home_money_change[i]=int(home.Close[0])-int(home.Open[0])
    game_data.away_money_close[i]=int(away.Close[0])
    if away.Open[0]=='NL':
        game_data.away_money_open[i]=game_data.away_money_close[i]
        game_data.away_money_change[i]=0
    else:
        game_data.away_money_open[i]=int(away.Open[0])
        game_data.away_money_change[i]=int(away.Close[0])-int(away.Open[0])
    print("in loop made it this far")
    game_data.home_prob_open[i]=home_pct_chance(game_data.home_money_open[i],game_data.away_money_open[i])
    game_data.home_prob_close[i]=home_pct_chance(game_data.home_money_close[i],game_data.away_money_close[i])
    game_data.home_prob_change[i]=game_data.home_prob_close[i]-game_data.home_prob_open[i]
    game_data.ou_open[i]=home.OpenOU[0]
    game_data.ou_close[i]=away.CloseOU[0]
    print("end of game_data loop")
'''

    except Exception as e:
        print("e in game_data loop", e)
        None
        '''
print("game_data debug", game_data)
if not game_data.empty:
    game_data=game_data.fillna(0)
    game_data=game_data[game_data.home_streak<1.001]
    game_data=game_data[game_data.away_streak<1.001]
    game_data=game_data[game_data.home_streak>-.001]
    game_data=game_data[game_data.away_streak>-.001]
    df=game_data.copy()
    home_dummies=pd.get_dummies(df.home_team)
    away_dummies=pd.get_dummies(df.away_team)
    home_dummies.columns=['h_'+str(col) for col in home_dummies.columns]
    away_dummies.columns=['a_'+str(col) for col in away_dummies.columns]
    df=df.merge(home_dummies,left_index=True,right_index=True)
    df=df.merge(away_dummies,left_index=True,right_index=True)
    df.drop(['date','lookback_days','home_team','away_team','home_starter_name',
             'away_starter_name','home_starter_id','away_starter_id','home_bat_team','home_bat_league',
            'away_bat_team','away_bat_league'],axis=1,inplace=True)
    # This moves the final moneyline to the last columns to make it easier to examine the real world application later on
    df['home_money_close2']=df.home_money_close
    df['away_money_close2']=df.away_money_close
    df.drop(['home_money_close','away_money_close'],axis=1,inplace=True)
    df['home_money_close']=df.home_money_close2
    df['away_money_close']=df.away_money_close2
    df.drop(['home_money_close2','away_money_close2'],axis=1,inplace=True)
    df.reset_index(drop=True,inplace=True)
    # Load existing data from the pickle file if it exists
    if os.path.exists("cleaned_data.pickle"):
        with open("cleaned_data.pickle", "rb") as pickle_in:
            existing_data = pickle.load(pickle_in)
    else:
        existing_data = pd.DataFrame()

    # Assuming df is the new data you want to append
    existing_data = pd.concat([existing_data, df], ignore_index=True)

    # Save the updated data back to the pickle file
    with open("cleaned_data.pickle", "wb") as pickle_out:
        pickle.dump(existing_data, pickle_out)

    print("cleaned_data.pickle", existing_data)

    '''
    pickle_out=open("cleaned_data.pickle","wb")
    pickle.dump(df,pickle_out)
    pickle_out.close()
    print("cleaned_data.pickle", df)
    '''



Found 27 files.
get_all_starters day 2010-04-04 00:00:00
get_all_starters game 277463
get_all_starters game 263816
get_all_starters day 2010-04-05 00:00:00
get_all_starters game 263822
get_all_starters game 263821
get_all_starters game 263820
get_all_starters game 263819
get_all_starters game 263818
get_all_starters game 263817
get_all_starters game 263815
get_all_starters game 263814
get_all_starters game 263813
get_all_starters game 263812
get_all_starters game 263811
get_all_starters game 263810
get_all_starters game 263809
get_all_starters day 2010-04-06 00:00:00
get_all_starters game 263837
get_all_starters game 263828
get_all_starters game 263827
get_all_starters game 263826
get_all_starters game 263825
get_all_starters game 263824
get_all_starters game 263823
get_all_starters day 2010-04-07 00:00:00
get_all_starters game 263851
get_all_starters game 263843
get_all_starters game 263842
get_all_starters game 263841
get_all_starters game 263840
get_all_starters game 263839
get_all_

get_all_starters game 264065
get_all_starters game 264064
get_all_starters game 264063
get_all_starters game 264062
get_all_starters game 264061
get_all_starters game 264060
get_all_starters game 264059
get_all_starters game 264058
get_all_starters game 264048
get_all_starters day 2010-04-25 00:00:00
get_all_starters game 264087
get_all_starters game 264086
get_all_starters game 264085
get_all_starters game 264084
get_all_starters game 264083
get_all_starters game 264082
get_all_starters game 264081
get_all_starters game 264080
get_all_starters game 264079
get_all_starters game 264078
get_all_starters game 264077
get_all_starters game 264076
get_all_starters game 264075
get_all_starters game 264074
get_all_starters game 264073
get_all_starters day 2010-04-26 00:00:00
get_all_starters game 264098
get_all_starters game 264097
get_all_starters game 264096
get_all_starters game 264095
get_all_starters game 264094
get_all_starters game 264093
get_all_starters game 264092
get_all_starters ga

Found 13 game files.
ATL CHC
home_schedule_record directory D:\BaseballBetsData1
home_schedule_record home_schedule_record_fn 2010_schedule_record_ATL.pkl
home_schedule_record full_path D:\BaseballBetsData1\2010_schedule_record_ATL.pkl
home_schedule_record found <_io.BufferedReader name='D:\\BaseballBetsData1\\2010_schedule_record_ATL.pkl'>
2010-04-05
2010-03-06
lookback_end_date_formatted 2010-03-05
data in loop 1 Empty DataFrame
Columns: []
Index: []
date_string2 Friday, Mar 5
home_schedule_record.index Index(['Monday, Apr 5', 'Wednesday, Apr 7', 'Thursday, Apr 8', 'Friday, Apr 9',
       'Saturday, Apr 10', 'Sunday, Apr 11', 'Monday, Apr 12',
       'Wednesday, Apr 14', 'Thursday, Apr 15', 'Friday, Apr 16',
       ...
       'Wednesday, Sep 22', 'Friday, Sep 24', 'Saturday, Sep 25',
       'Sunday, Sep 26', 'Monday, Sep 27', 'Tuesday, Sep 28',
       'Wednesday, Sep 29', 'Friday, Oct 1', 'Saturday, Oct 2',
       'Sunday, Oct 3'],
      dtype='object', name='Date', length=162)
data 

data in loop 7    home_win  home_score  away_score       date lookback_days home_team  \
0         1        16.0         5.0 2010-04-05            30       ATL   

  away_team  home_pct  away_pct  home_streak  ...  home_starter_id  \
0       CHC         0         0            0  ...           117955   

  away_starter_id home_starter_launch  home_starter_est_ba_sa  \
0          407296                 NaN                     NaN   

   home_starter_est_woba_sa  home_starter_sum_woba  away_starter_launch  \
0                       NaN                    0.0                  NaN   

   away_starter_est_ba_sa  away_starter_est_woba_sa  away_starter_sum_woba  
0                     NaN                       NaN                    0.0  

[1 rows x 23 columns]
data in loop 2    home_win  home_score  away_score       date lookback_days home_team  \
0         1        16.0         5.0 2010-04-05            30       ATL   

  away_team  home_pct  away_pct  home_streak  ...  home_starter_id  \
0 

NYM MIA
home_schedule_record directory D:\BaseballBetsData1
home_schedule_record home_schedule_record_fn 2010_schedule_record_NYM.pkl
home_schedule_record full_path D:\BaseballBetsData1\2010_schedule_record_NYM.pkl
home_schedule_record found <_io.BufferedReader name='D:\\BaseballBetsData1\\2010_schedule_record_NYM.pkl'>
PIT LAD
home_schedule_record directory D:\BaseballBetsData1
home_schedule_record home_schedule_record_fn 2010_schedule_record_PIT.pkl
home_schedule_record full_path D:\BaseballBetsData1\2010_schedule_record_PIT.pkl
home_schedule_record found <_io.BufferedReader name='D:\\BaseballBetsData1\\2010_schedule_record_PIT.pkl'>
2010-04-05
2010-03-06
lookback_end_date_formatted 2010-03-05
data in loop 1    home_win  home_score  away_score       date lookback_days home_team  \
0         1        16.0         5.0 2010-04-05            30       ATL   
1         1         6.0         0.0 2010-04-05            30       CHW   
2         0         4.0         8.0 2010-04-05            

...

LAD PIT
home_schedule_record directory D:\BaseballBetsData1
home_schedule_record home_schedule_record_fn 2010_schedule_record_LAD.pkl
home_schedule_record full_path D:\BaseballBetsData1\2010_schedule_record_LAD.pkl
home_schedule_record found <_io.BufferedReader name='D:\\BaseballBetsData1\\2010_schedule_record_LAD.pkl'>
2010-04-30
2010-03-31
lookback_end_date_formatted 2010-03-30
data in loop 1    home_win  home_score  away_score       date lookback_days home_team  \
0         1         5.0         4.0 2010-04-30            30       BAL   
1         1         4.0         2.0 2010-04-30            30       ATL   
2         0         2.0         3.0 2010-04-30            30        TB   
3         1         3.0         0.0 2010-04-30            30        SD   
4         0         1.0         9.0 2010-04-30            30       PHI   

  away_team  home_pct  away_pct  home_streak  ...  home_starter_id  \
0       BOS         0         0            0  ...           456696   
1       HOU      

data in loop 7    home_win  home_score  away_score       date lookback_days home_team  \
0         1         5.0         4.0 2010-04-30            30       BAL   
1         1         4.0         2.0 2010-04-30            30       ATL   
2         0         2.0         3.0 2010-04-30            30        TB   
3         1         3.0         0.0 2010-04-30            30        SD   
4         0         1.0         9.0 2010-04-30            30       PHI   
5         1         6.0         2.0 2010-04-30            30       LAD   

  away_team  home_pct  away_pct  home_streak  ...  home_starter_id  \
0       BOS         0         0            0  ...           456696   
1       HOU         0         0            0  ...           462102   
2        KC         0         0            0  ...           435298   
3       MIL         0         0            0  ...           453385   
4       NYM         0         0            0  ...           452718   
5       PIT         0         0            0  

data in loop 3    home_win  home_score  away_score       date lookback_days home_team  \
0         1         5.0         4.0 2010-04-30            30       BAL   
1         1         4.0         2.0 2010-04-30            30       ATL   
2         0         2.0         3.0 2010-04-30            30        TB   
3         1         3.0         0.0 2010-04-30            30        SD   
4         0         1.0         9.0 2010-04-30            30       PHI   
5         1         6.0         2.0 2010-04-30            30       LAD   
6         0         0.0         2.0 2010-04-30            30       SEA   

  away_team  home_pct  away_pct  home_streak  ...  home_starter_id  \
0       BOS         0         0            0  ...           456696   
1       HOU         0         0            0  ...           462102   
2        KC         0         0            0  ...           435298   
3       MIL         0         0            0  ...           453385   
4       NYM         0         0           

SF COL
home_schedule_record directory D:\BaseballBetsData1
home_schedule_record home_schedule_record_fn 2010_schedule_record_SF.pkl
home_schedule_record full_path D:\BaseballBetsData1\2010_schedule_record_SF.pkl
home_schedule_record found <_io.BufferedReader name='D:\\BaseballBetsData1\\2010_schedule_record_SF.pkl'>
2010-04-30
2010-03-31
lookback_end_date_formatted 2010-03-30
data in loop 1    home_win  home_score  away_score       date lookback_days home_team  \
0         1         5.0         4.0 2010-04-30            30       BAL   
1         1         4.0         2.0 2010-04-30            30       ATL   
2         0         2.0         3.0 2010-04-30            30        TB   
3         1         3.0         0.0 2010-04-30            30        SD   
4         0         1.0         9.0 2010-04-30            30       PHI   
5         1         6.0         2.0 2010-04-30            30       LAD   
6         0         0.0         2.0 2010-04-30            30       SEA   
7         1   

this ran game_data      home_win  home_score  away_score       date lookback_days home_team  \
0           1        16.0         5.0 2010-04-05            30       ATL   
1           1         6.0         0.0 2010-04-05            30       CHW   
2           0         4.0         8.0 2010-04-05            30        KC   
3           1        11.0         5.0 2010-04-05            30       PIT   
4           1         6.0         3.0 2010-04-05            30       LAA   
..        ...         ...         ...        ...           ...       ...   
253         1         6.0         2.0 2010-04-30            30       LAD   
254         0         0.0         2.0 2010-04-30            30       SEA   
255         1        10.0         2.0 2010-04-30            30       TOR   
256         0         2.0         3.0 2010-04-30            30       STL   
257         1         5.0         2.0 2010-04-30            30        SF   

    away_team  home_pct  away_pct  home_streak  ...  home_money_clos

cleaned_data.pickle      home_win  home_score  away_score  home_pct  away_pct  home_streak  \
0           0         3.0         4.0  0.000000  0.000000          0.0   
1           1         7.0         0.0  0.000000  0.000000          0.0   
2           0         5.0         6.0  0.000000  0.000000          0.0   
3           0         3.0        10.0  0.000000  0.000000          0.0   
4           1         7.0         1.0  0.000000  0.000000          0.0   
..        ...         ...         ...       ...       ...          ...   
264         1         4.0         3.0  1.000000  1.000000         -0.0   
265         0         3.0        10.0  0.333333  0.666667         -0.0   
266         0         4.0         6.0  0.500000  0.000000          1.0   
267         0         0.0        11.0  0.500000  0.333333          1.0   
268         0         1.0         3.0  1.000000  0.333333          1.0   

     away_streak  home_starter_launch  home_starter_est_ba_sa  \
0            0.0          

'\npickle_out=open("cleaned_data.pickle","wb")\npickle.dump(df,pickle_out)\npickle_out.close()\nprint("cleaned_data.pickle", df)\n'

In [21]:
df = open("cleaned_data.pickle","rb")
df=pickle.load(df)
print(df.columns)
#print(df[df['home_money_close'] != 0]['home_money_close'])
'''
for column in df.columns:
    if df[column].nunique() == 1:
        print(f"Column '{column}' has all the same values.")
    else:
        print(f"Column '{column}' does not have all the same values.")
'''

columns_to_check = ['home_bat_est_ba_sa', 'home_bat_est_woba_sa', 'home_bat_sum_woba', 'away_bat_est_ba_sa',
                   'away_bat_est_woba_sa', 'away_bat_sum_woba', 'homepen_est_ba_sa', 'homepen_est_woba_sa',
                   'homepen_sum_woba', 'awaypen_est_ba_sa', 'awaypen_est_woba_sa', 'awaypen_sum_woba']
exists = df.columns.isin(columns_to_check).any()
if exists:
    print("At least one of the columns exists.")
else:
    print("None of the columns exist.")

'''
print(df['home_bat_est_ba_sa'])
print(df['home_bat_est_woba_sa'])
print(df['home_bat_sum_woba'])
print(df['away_bat_est_ba_sa'])
print(df['away_bat_est_woba_sa'])
print(df['away_bat_sum_woba'])
print(df['homepen_est_ba_sa'])
print(df['homepen_est_woba_sa'])
print(df['homepen_sum_woba'])
print(df['awaypen_est_ba_sa'])
print(df['awaypen_est_woba_sa'])
print(df['awaypen_sum_woba'])
'''
df.sample(30)

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_data.pickle'

In [11]:
df.describe()
len(df)

269

In [10]:
df.dtypes

home_win              int64
home_score          float64
away_score          float64
home_pct            float64
away_pct            float64
                     ...   
a_TEX                  bool
a_TOR                  bool
a_WSN                  bool
home_money_close      int64
away_money_close      int64
Length: 216, dtype: object